# 06. AI Support Resolution Desk

This notebook is the flagship ML-facing AETHER application pack.

We will model one governed support desk where:

- a customer issue lands in one replayable history
- artifact and vector sidecars retrieve useful support evidence
- planner logic publishes candidate resolutions and escalations
- AETHER derives which path is actually ready
- assignment authority moves through a live handoff
- stale recommendations are fenced
- one tuple explanation shows why the current selected path is valid


In [ ]:
from pathlib import Path
import subprocess
import sys
from textwrap import dedent

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path
        for path in candidate_roots
        if (path / "python").exists() and (path / "Cargo.toml").exists()
    ),
    None,
)

if repo_root is None:
    repo_root = Path("/content/AETHER")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fyremael/AETHER", str(repo_root)],
            check=True,
        )

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

repo_root


In [ ]:
from notebooks.colab_setup import ensure_rust_toolchain, pretty_json, start_http_service, stop_http_service
from aether_sdk import (
    AetherClient,
    make_artifact_reference,
    make_datom,
    make_vector_record,
    value_entity,
    value_string,
    value_u64,
)

ensure_rust_toolchain()
session = start_http_service(repo_root)
client = AetherClient(session.base_url)

pretty_json(client.health())


## Seed The Support Desk History

These datoms model one support case, one completed prerequisite step, and two candidate resolution paths.


In [ ]:
support_history = [
    make_datom(entity=501, attribute=1, value=value_string("duplicate charge after plan migration"), element=1),
    make_datom(entity=501, attribute=2, value=value_string("high"), element=2),
    make_datom(entity=501, attribute=3, value=value_string("chat"), element=3),
    make_datom(entity=801, attribute=4, value=value_string("collect billing timeline"), element=4),
    make_datom(entity=801, attribute=5, value=value_string("done"), element=5),
    make_datom(entity=901, attribute=6, value=value_string("apply-migration-credit"), element=6),
    make_datom(entity=901, attribute=7, value=value_entity(501), element=7),
    make_datom(entity=901, attribute=8, value=value_entity(801), element=8, op="Add"),
    make_datom(entity=901, attribute=9, value=value_string("approved"), element=9),
    make_datom(entity=901, attribute=10, value=value_string("clear"), element=10),
    make_datom(entity=901, attribute=11, value=value_string("high"), element=11),
    make_datom(entity=902, attribute=6, value=value_string("escalate-to-billing-specialist"), element=12),
    make_datom(entity=902, attribute=7, value=value_entity(501), element=13),
    make_datom(entity=902, attribute=9, value=value_string("approved"), element=14),
    make_datom(entity=902, attribute=10, value=value_string("suppressed"), element=15),
    make_datom(entity=902, attribute=11, value=value_string("medium"), element=16),
    make_datom(entity=501, attribute=15, value=value_string("open"), element=17),
    make_datom(entity=501, attribute=16, value=value_string("support-memory-anchor-1"), element=19, op="Annotate"),
]

client.append(support_history)
pretty_json(client.history())


## Register Retrieved Evidence

The support-memory sidecar stays subordinate to the journal, so the artifact and vector records are anchored to real journal elements.


In [ ]:
client.register_artifact_reference(
    make_artifact_reference(
        sidecar_id="support-memory",
        artifact_id="kb-apply-credit",
        entity=9101,
        uri="s3://aether/support/runbooks/migration-credit.md",
        media_type="text/markdown",
        byte_length=1824,
        digest="sha256:kb-apply-credit",
        metadata={"kind": {"String": "runbook"}, "title": {"String": "Apply migration credit"}},
        registered_at=19,
    )
)

client.append(
    [
        make_datom(entity=501, attribute=16, value=value_string("support-memory-anchor-2"), element=20, op="Annotate"),
    ]
)

client.register_vector_record(
    record=make_vector_record(
        sidecar_id="support-memory",
        vector_id="vec-apply-credit",
        entity=9101,
        source_artifact_id="kb-apply-credit",
        embedding_ref="s3://aether/support/vectors/vec-apply-credit.bin",
        dimensions=3,
        metric="cosine",
        metadata={"topic": {"String": "migration-credit"}},
        registered_at=20,
    ),
    embedding=[0.96, 0.04, 0.01],
)

search = client.search_vectors(
    {
        "sidecar_id": "support-memory",
        "query_embedding": [1.0, 0.0, 0.0],
        "top_k": 1,
        "metric": "cosine",
        "as_of": 20,
        "projection": {
            "predicate": {
                "id": 81,
                "name": "retrieved_support_evidence",
                "arity": 3,
            },
            "query_entity": 501,
        },
    }
)

pretty_json(search)


In [ ]:
client.append(
    [
        make_datom(entity=501, attribute=12, value=value_string("triage-agent"), element=21),
        make_datom(entity=501, attribute=13, value=value_u64(1), element=22),
        make_datom(entity=501, attribute=14, value=value_string("active"), element=23),
        make_datom(entity=501, attribute=12, value=value_string("lead-ana"), element=24),
        make_datom(entity=501, attribute=13, value=value_u64(2), element=25),
    ]
)

pretty_json(client.history())


In [ ]:
def render_fact_value(value):
    if "Entity" in value:
        return f"entity({value['Entity']})"
    if "String" in value:
        escaped = value["String"].replace("\\", "\\\\").replace('"', '\\"')
        return f'"{escaped}"'
    if "U64" in value:
        return str(value["U64"])
    if "F64" in value:
        return f"{value['F64']:.4f}"
    raise ValueError(f"unsupported value shape: {value}")


def render_projected_facts(search_response):
    lines = []
    for fact in search_response["facts"]:
        values = ", ".join(render_fact_value(value) for value in fact["values"])
        lines.append(f"  {fact['predicate']['name']}({values})")
    return "\n".join(lines)


projected_facts = render_projected_facts(search)
print(projected_facts)


In [ ]:
def support_document(view: str, goal: str) -> str:
    return dedent(
        f"""
        schema v1 {{
          attr case.subject: ScalarLWW<String>
          attr case.priority: ScalarLWW<String>
          attr case.channel: ScalarLWW<String>
          attr step.title: ScalarLWW<String>
          attr step.status: ScalarLWW<String>
          attr resolution.title: ScalarLWW<String>
          attr resolution.case: RefScalar<Entity>
          attr resolution.depends_on: RefSet<Entity>
          attr resolution.approval_state: ScalarLWW<String>
          attr resolution.suppression_state: ScalarLWW<String>
          attr resolution.confidence_band: ScalarLWW<String>
          attr case.claimed_by: ScalarLWW<String>
          attr case.lease_epoch: ScalarLWW<U64>
          attr case.lease_state: ScalarLWW<String>
          attr case.status: ScalarLWW<String>
          attr system.anchor: ScalarLWW<String>
        }}

        predicates {{
          support_case(Entity)
          candidate_resolution(Entity)
          assignment_attempt(Entity, String, U64)
          retrieved_support_evidence(Entity, Entity, F64)
          case_subject(Entity, String)
          case_priority(Entity, String)
          case_channel(Entity, String)
          step_status(Entity, String)
          resolution_title(Entity, String)
          resolution_case(Entity, Entity)
          resolution_depends_on(Entity, Entity)
          resolution_approval_state(Entity, String)
          resolution_suppression_state(Entity, String)
          resolution_confidence_band(Entity, String)
          case_claimed_by(Entity, String)
          case_lease_epoch(Entity, U64)
          case_lease_state(Entity, String)
          case_status(Entity, String)
          active_case(Entity, String, String, String)
          resolution_dependency_closure(Entity, Entity)
          step_complete(Entity)
          resolution_blocked(Entity)
          resolution_policy_approved(Entity)
          resolution_suppressed(Entity)
          resolution_confident(Entity)
          resolution_has_retrieved_evidence(Entity)
          active_assignment(Entity, String, U64)
          case_claimed(Entity)
          resolution_board(Entity, String, String, String, String)
          case_action_ready(Entity)
          ready_resolution_detail(Entity, String, Entity, String)
          case_resolution_selected(Entity, Entity, String, U64)
          case_resolution_selected_detail(Entity, String, String, String, U64)
          stale_assignment_attempt(Entity, String, U64)
          stale_assignment_attempt_detail(Entity, String, String, U64)
        }}

        facts {{
          support_case(entity(501))
          candidate_resolution(entity(901))
          candidate_resolution(entity(902))
          assignment_attempt(entity(501), "triage-agent", 1)
          assignment_attempt(entity(501), "lead-ana", 1)
          assignment_attempt(entity(501), "triage-agent", 2)
          assignment_attempt(entity(501), "lead-ana", 2)
    f"{projected_facts}\n",
        }}

        rules {{
          active_case(case, subject, priority, channel) <- support_case(case), case_status(case, "open"), case_subject(case, subject), case_priority(case, priority), case_channel(case, channel)
          resolution_dependency_closure(resolution, dep) <- resolution_depends_on(resolution, dep)
          resolution_dependency_closure(resolution, dep) <- resolution_depends_on(resolution, mid), resolution_dependency_closure(mid, dep)
          step_complete(step) <- step_status(step, "done")
          resolution_blocked(resolution) <- resolution_dependency_closure(resolution, dep), not step_complete(dep)
          resolution_policy_approved(resolution) <- resolution_approval_state(resolution, "approved")
          resolution_suppressed(resolution) <- resolution_suppression_state(resolution, "suppressed")
          resolution_confident(resolution) <- resolution_confidence_band(resolution, "high")
          resolution_has_retrieved_evidence(resolution) <- resolution_case(resolution, case), retrieved_support_evidence(case, evidence, score)
          active_assignment(case, owner, epoch) <- case_claimed_by(case, owner), case_lease_epoch(case, epoch), case_lease_state(case, "active")
          case_claimed(case) <- active_assignment(case, owner, epoch)
          resolution_board(resolution, title, approval, suppression, confidence) <- candidate_resolution(resolution), resolution_title(resolution, title), resolution_approval_state(resolution, approval), resolution_suppression_state(resolution, suppression), resolution_confidence_band(resolution, confidence)
          case_action_ready(resolution) <- candidate_resolution(resolution), resolution_policy_approved(resolution), resolution_confident(resolution), resolution_has_retrieved_evidence(resolution), not resolution_blocked(resolution), not resolution_suppressed(resolution), resolution_case(resolution, case), not case_claimed(case)
          ready_resolution_detail(case, subject, resolution, title) <- case_action_ready(resolution), resolution_case(resolution, case), case_subject(case, subject), resolution_title(resolution, title)
          case_resolution_selected(resolution, case, owner, epoch) <- candidate_resolution(resolution), resolution_policy_approved(resolution), resolution_confident(resolution), resolution_has_retrieved_evidence(resolution), not resolution_blocked(resolution), not resolution_suppressed(resolution), resolution_case(resolution, case), active_assignment(case, owner, epoch)
          case_resolution_selected_detail(case, subject, title, owner, epoch) <- case_resolution_selected(resolution, case, owner, epoch), case_subject(case, subject), resolution_title(resolution, title)
          stale_assignment_attempt(case, owner, epoch) <- assignment_attempt(case, owner, epoch), not active_assignment(case, owner, epoch)
          stale_assignment_attempt_detail(case, subject, owner, epoch) <- stale_assignment_attempt(case, owner, epoch), case_subject(case, subject)
        }}

        materialize {
          active_case
          resolution_dependency_closure
          resolution_blocked
          resolution_policy_approved
          resolution_suppressed
          resolution_confident
          resolution_has_retrieved_evidence
          active_assignment
          case_claimed
          resolution_board
          case_action_ready
          ready_resolution_detail
          case_resolution_selected
          case_resolution_selected_detail
          stale_assignment_attempt
          stale_assignment_attempt_detail
        }

        query current_cut {
    f"          {view}\n",
    f"          {goal}\n",
        }
        """
    ).strip()


In [ ]:
active_cases = client.run_document(
    support_document(
        "current",
        "goal active_case(case, subject, priority, channel)\n  keep case, subject, priority, channel",
    )
)
pretty_json(active_cases["query"])


In [ ]:
candidate_resolutions = client.run_document(
    support_document(
        "current",
        "goal resolution_board(resolution, title, approval, suppression, confidence)\n  keep resolution, title, approval, suppression, confidence",
    )
)
pretty_json(candidate_resolutions["query"])


In [ ]:
ready_resolution = client.run_document(
    support_document(
        "as_of e20",
        "goal ready_resolution_detail(case, subject, resolution, title)\n  keep case, subject, resolution, title",
    )
)
pretty_json(ready_resolution["query"])


In [ ]:
current_selection = client.run_document(
    support_document(
        "current",
        "goal case_resolution_selected_detail(case, subject, title, owner, epoch)\n  keep case, subject, title, owner, epoch",
    )
)
pretty_json(current_selection["query"])

selected_tuple_id = current_selection["query"]["rows"][0]["tuple_id"]
pretty_json(client.explain_tuple(selected_tuple_id))


In [ ]:
before_handoff = client.run_document(
    support_document(
        "as_of e23",
        "goal case_resolution_selected_detail(case, subject, title, owner, epoch)\n  keep case, subject, title, owner, epoch",
    )
)
stale_attempts = client.run_document(
    support_document(
        "current",
        "goal stale_assignment_attempt_detail(case, subject, owner, epoch)\n  keep case, subject, owner, epoch",
    )
)

pretty_json(before_handoff["query"])
pretty_json(stale_attempts["query"])


## What This Proves

- AETHER can host an ML-relevant support workflow without inventing new core semantics
- retrieval can re-enter the case flow as governed evidence rather than floating beside it
- assignment handoff, stale fencing, replay, and proof still hold in the end-user app story

That is the current posture: one concrete support application pack built honestly on the v1 pilot proof.


In [ ]:
# Uncomment this cell when you are done with the notebook.
# stop_http_service(session)
